#### Import Dataset

In [ ]:
import pandas as pd

df = pd.read_excel("Irish Hotel Reviews.xlsx", sheet_name="Data")

print(df.shape)
print(df.columns)

(13500, 21)
Index(['review_id', 'review_title', 'review_text', 'review_tip',
       'overall_rating', 'value_rating', 'rooms_rating', 'location_rating',
       'cleanliness_rating', 'service_rating', 'sleep_quality_rating',
       'language', 'trip_type', 'stay_date', 'reviewer_name',
       'reviewer_username', 'contribution_count', 'published_date',
       'review_link', 'hotel_name', 'county'],
      dtype='object')


#### Select Required Columns

In [ ]:
df = df[
    [
        "review_id",
        "review_text",
        "overall_rating",
        "hotel_name",
        "county",
        "trip_type",
        "language"
    ]
]

In [ ]:
df.head()

,review_id,review_text,overall_rating,hotel_name,county,trip_type,language
0,1062725118,This hotel was a great location. You can walk ...,4,Academy Plaza Hotel,Dublin,friends,en
1,1062581658,Dublin hotels are quite expensive and this one...,4,Academy Plaza Hotel,Dublin,couples,en
2,1060339977,"The location of the hotel is very good, centra...",3,Academy Plaza Hotel,Dublin,solo,en
3,1060065498,"A hotel very well located, with very pleasant ...",5,Academy Plaza Hotel,Dublin,couples,en
4,1059356019,I stayed in this hotel many times the year has...,5,Academy Plaza Hotel,Dublin,business,en


#### Remove duplicates

In [ ]:
before = len(df)

df = df.drop_duplicates(subset=["review_text"])

after = len(df)

print("Removed:", before-after)

Removed: 206


#### Check for missing values

In [ ]:
print(df.isnull().sum())

review_id            0
review_text          0
overall_rating       0
hotel_name           0
county               0
trip_type         1715
language             0
dtype: int64


#### Count language values

In [ ]:
print(df["language"].value_counts())

language
en    13294
Name: count, dtype: int64


#### Clean the text

In [ ]:
import re

def clean_text(text):

    text = str(text)

    # Remove URLs
    text = re.sub(r"http\S+", "", text)

    # Remove HTML
    text = re.sub(r"<.*?>", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Remove strange symbols
    text = re.sub(r"[^\w\s.,!?]", "", text)

    return text.strip()

In [ ]:
df["clean_review"] = df["review_text"].apply(clean_text)

In [ ]:
df.head()

,review_id,review_text,overall_rating,hotel_name,county,trip_type,language,clean_review
0,1062725118,This hotel was a great location. You can walk ...,4,Academy Plaza Hotel,Dublin,friends,en,This hotel was a great location. You can walk ...
1,1062581658,Dublin hotels are quite expensive and this one...,4,Academy Plaza Hotel,Dublin,couples,en,Dublin hotels are quite expensive and this one...
2,1060339977,"The location of the hotel is very good, centra...",3,Academy Plaza Hotel,Dublin,solo,en,"The location of the hotel is very good, centra..."
3,1060065498,"A hotel very well located, with very pleasant ...",5,Academy Plaza Hotel,Dublin,couples,en,"A hotel very well located, with very pleasant ..."
4,1059356019,I stayed in this hotel many times the year has...,5,Academy Plaza Hotel,Dublin,business,en,I stayed in this hotel many times the year has...


#### Review Length Analysis

In [ ]:
df["word_count"] = df["clean_review"].apply(
    lambda x: len(x.split())
)

In [ ]:
print(df["word_count"].describe())

count    13294.000000
mean        72.137130
std         73.166436
min          3.000000
25%         32.000000
50%         50.000000
75%         85.000000
max       1474.000000
Name: word_count, dtype: float64


#### Create Sentiment Labels using Overall Rating

In [ ]:
def create_sentiment(rating):

    if rating >= 4:
        return "positive"

    elif rating == 3:
        return "neutral"

    else:
        return "negative"

In [ ]:
df["sentiment"] = df["overall_rating"].apply(create_sentiment)

In [ ]:
print(df["sentiment"].value_counts())

sentiment
positive    12313
negative      564
neutral       417
Name: count, dtype: int64


#### NER Anonymization

In [ ]:
pip install spacy

In [ ]:
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [ ]:
def anonymize_text(text):

    doc = nlp(text)

    new_text = text

    for ent in doc.ents:

        if ent.label_ == "PERSON":
            new_text = new_text.replace(
                ent.text,
                "[PERSON]"
            )

    return new_text

In [ ]:
df["anonymized_review"] = (
    df["clean_review"]
    .apply(anonymize_text)
)

In [ ]:
df.head()

,review_id,review_text,overall_rating,hotel_name,county,trip_type,language,clean_review,word_count,sentiment,anonymized_review
0,1062725118,This hotel was a great location. You can walk ...,4,Academy Plaza Hotel,Dublin,friends,en,This hotel was a great location. You can walk ...,119,positive,This hotel was a great location. You can walk ...
1,1062581658,Dublin hotels are quite expensive and this one...,4,Academy Plaza Hotel,Dublin,couples,en,Dublin hotels are quite expensive and this one...,76,positive,Dublin hotels are quite expensive and this one...
2,1060339977,"The location of the hotel is very good, centra...",3,Academy Plaza Hotel,Dublin,solo,en,"The location of the hotel is very good, centra...",226,neutral,"The location of the hotel is very good, centra..."
3,1060065498,"A hotel very well located, with very pleasant ...",5,Academy Plaza Hotel,Dublin,couples,en,"A hotel very well located, with very pleasant ...",11,positive,"A hotel very well located, with very pleasant ..."
4,1059356019,I stayed in this hotel many times the year has...,5,Academy Plaza Hotel,Dublin,business,en,I stayed in this hotel many times the year has...,111,positive,I stayed in this hotel many times the year has...


#### Remove hotel names

In [ ]:
def remove_hotel_name(row):

    review = row["anonymized_review"]

    hotel = str(row["hotel_name"])

    review = review.replace(hotel, "[HOTEL]")

    return review

In [ ]:
df["final_review"] = df.apply(
    remove_hotel_name,
    axis=1
)

In [ ]:
df.head()

,review_id,review_text,overall_rating,hotel_name,county,trip_type,language,clean_review,word_count,sentiment,anonymized_review,final_review
0,1062725118,This hotel was a great location. You can walk ...,4,Academy Plaza Hotel,Dublin,friends,en,This hotel was a great location. You can walk ...,119,positive,This hotel was a great location. You can walk ...,This hotel was a great location. You can walk ...
1,1062581658,Dublin hotels are quite expensive and this one...,4,Academy Plaza Hotel,Dublin,couples,en,Dublin hotels are quite expensive and this one...,76,positive,Dublin hotels are quite expensive and this one...,Dublin hotels are quite expensive and this one...
2,1060339977,"The location of the hotel is very good, centra...",3,Academy Plaza Hotel,Dublin,solo,en,"The location of the hotel is very good, centra...",226,neutral,"The location of the hotel is very good, centra...","The location of the hotel is very good, centra..."
3,1060065498,"A hotel very well located, with very pleasant ...",5,Academy Plaza Hotel,Dublin,couples,en,"A hotel very well located, with very pleasant ...",11,positive,"A hotel very well located, with very pleasant ...","A hotel very well located, with very pleasant ..."
4,1059356019,I stayed in this hotel many times the year has...,5,Academy Plaza Hotel,Dublin,business,en,I stayed in this hotel many times the year has...,111,positive,I stayed in this hotel many times the year has...,I stayed in this hotel many times the year has...


#### Preprocess data for traditional ML models

In [ ]:
pip install nltk

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [ ]:
import re

def preprocess_for_ml(text):

    # lowercase
    text = text.lower()

    # keep only letters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # tokenize
    tokens = word_tokenize(text)

    # remove stop words
    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    # lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [ ]:
df["clean_review_ml"] = (
    df["final_review"]
    .apply(preprocess_for_ml)
)

In [ ]:
df.head()

,review_id,review_text,overall_rating,hotel_name,county,trip_type,language,clean_review,word_count,sentiment,anonymized_review,final_review,clean_review_ml
0,1062725118,This hotel was a great location. You can walk ...,4,Academy Plaza Hotel,Dublin,friends,en,This hotel was a great location. You can walk ...,119,positive,This hotel was a great location. You can walk ...,This hotel was a great location. You can walk ...,hotel great location walk room spacious europe...
1,1062581658,Dublin hotels are quite expensive and this one...,4,Academy Plaza Hotel,Dublin,couples,en,Dublin hotels are quite expensive and this one...,76,positive,Dublin hotels are quite expensive and this one...,Dublin hotels are quite expensive and this one...,dublin hotel quite expensive one affordable ye...
2,1060339977,"The location of the hotel is very good, centra...",3,Academy Plaza Hotel,Dublin,solo,en,"The location of the hotel is very good, centra...",226,neutral,"The location of the hotel is very good, centra...","The location of the hotel is very good, centra...",location hotel good central quiet side street ...
3,1060065498,"A hotel very well located, with very pleasant ...",5,Academy Plaza Hotel,Dublin,couples,en,"A hotel very well located, with very pleasant ...",11,positive,"A hotel very well located, with very pleasant ...","A hotel very well located, with very pleasant ...",hotel well located pleasant available staff
4,1059356019,I stayed in this hotel many times the year has...,5,Academy Plaza Hotel,Dublin,business,en,I stayed in this hotel many times the year has...,111,positive,I stayed in this hotel many times the year has...,I stayed in this hotel many times the year has...,stayed hotel many time year passed since amaze...


In [ ]:
df[
 [
   "final_review",
   "clean_review_ml"
 ]
].head()

,final_review,clean_review_ml
0,This hotel was a great location. You can walk ...,hotel great location walk room spacious europe...
1,Dublin hotels are quite expensive and this one...,dublin hotel quite expensive one affordable ye...
2,"The location of the hotel is very good, centra...",location hotel good central quiet side street ...
3,"A hotel very well located, with very pleasant ...",hotel well located pleasant available staff
4,I stayed in this hotel many times the year has...,stayed hotel many time year passed since amaze...


In [ ]:
df.to_csv(
    "irish_hotel_reviews_processed.csv",
    index=False
)